## *HR Resume Screening Application with RAG*
### *Configuration & Setup*

In [ ]:
# Install required packages
!pip install -q langchain openai faiss-cpu pymupdf python-docx pandas tiktoken sentence-transformers langchain-community rank_bm25

### Import necessary libraries

In [ ]:
import os
import json
import glob
import re
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import faiss
import fitz  # PyMuPDF
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

from google.colab import userdata, files

# === LangChain Core ===
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.agents import Tool, AgentExecutor, LLMSingleActionAgent
from langchain.memory import ConversationBufferMemory
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain.docstore.document import Document

# === LangChain Models & Embeddings ===
from langchain_community.chat_models import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# === Sentence Transformers (local embeddings) ===
from sentence_transformers import SentenceTransformer

# === Document loaders ===
from langchain.document_loaders import PyMuPDFLoader, UnstructuredWordDocumentLoader

# === BM25 for Sparse Retrieval ===
from rank_bm25 import BM25Okapi

### Initialize global variables

In [ ]:
# Load secrets / tokens
openai_api_key = userdata.get("GPT_OSS_API")
hf_token = userdata.get("HF_TOKEN")
hr_smtp_gmail = userdata.get("HR_SMTP_GMAIL")

# === Config ===
cvs_folder = "/content/CVs"
embedding_model_name = "all-MiniLM-L6-v2"

In [ ]:
# Load CV files dynamically
cv_files = glob.glob(os.path.join(cvs_folder, "*.pdf"))

In [ ]:
# Sample Job Description (test query)
job_description = (
    "Junior Generative AI Engineer with solid Python skills, basic ML/DL knowledge, and familiarity with PyTorch or TensorFlow.\n"
    "Should understand LLMs and generative models, work with APIs and cloud deployment, and be eager to learn.\n"
    "Plus if familiar with LangChain, Hugging Face, and vector databases (FAISS, Pinecone). Location: Cairo or Remote.\n"
)

### Create CVs directory

In [ ]:
Path(cvs_folder).mkdir(exist_ok=True)

### Data Ingestion & Context-Aware Summarization

In [ ]:
# === CV Text Extraction ===
def extract_text_from_cv(file_path: str) -> str:
    """Simple and reliable text extraction from PDF (only PDF supported for now)."""
    if not file_path.endswith('.pdf'):
        raise ValueError("Only PDF files are supported for now.")

    try:
        text = ""
        with fitz.open(file_path) as pdf:
            for page in pdf:
                text += page.get_text("text")
        return text.strip()
    except Exception as e:
        print(f"Failed to extract text from {file_path}: {e}")
        return ""

In [ ]:
cv_text = extract_text_from_cv(file_path=cv_files[1])
cv_text

'Mahmoud Nasser\nGenerative AI & NLP Engineer\nmahmoud.nasser.abdelfatah@gmail.com\n \n01119065308\n \nGiza, Egypt\n \nwww.linkedin.com/in/mhnasser\n \ngithub.com/mahmoud0nasser\n \nPROFILE\nAI Developer & Machine Learning Engineer with hands-on experience in building intelligent systems using \nLLMs, GANs, and deep learning techniques. Skilled in Python, NLP, computer vision, and real-time AI \nintegrations. Developed and deployed full-stack AI applications including chatbot agents, speech-to-text \nsystems, and text-to-image models. Strong background in prompt engineering, MLOps, and statistical \nmodeling. Passionate about solving real-world problems through scalable, intelligent solutions.\nEDUCATION\nFaculty of Computers and Artificial Intelligence\nHelwan University\n•Major: Artificial Intelligence \n09/2021 – 06/2025\nCairo, Egypt\n•General Grade: Very Good\n•Graduation Project: Quranic ASR System (Whisper + NLP, Arabic Phonetics) – \nBuilt full-stack e-learning platform with ML

In [ ]:
def generate_candidate_profile(cv_text: str, cv_filename: str):
    """Parse CV text and generate a structured candidate profile as JSON."""
    clean_text = re.sub(r'\s+', ' ', cv_text).strip()
    if len(clean_text) > 7000:
        clean_text = clean_text[:7000]  # limit context for LLM

    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-oss-120b",
        temperature=0,
        openai_api_key=openai_api_key
    )

    response_schemas = [
        ResponseSchema(name="name", description="Full name"),
        ResponseSchema(name="email", description="Email address"),
        ResponseSchema(name="phone", description="Phone number"),
        ResponseSchema(name="location", description="City, Country"),
        ResponseSchema(name="years_experience", description="Number of years of experience (integer)"),
        ResponseSchema(name="skills", description="List of technical/professional skills"),
        ResponseSchema(name="education", description="List of educational qualifications (degrees, universities)"),
        ResponseSchema(name="key_projects", description="List of key projects or work experiences with brief descriptions"),
        ResponseSchema(name="validation_status", description='"complete" if name and email are provided, otherwise "incomplete"'),
    ]

    parser = StructuredOutputParser.from_response_schemas(response_schemas)
    format_instructions = parser.get_format_instructions()

    prompt = ChatPromptTemplate.from_template("""
    You are an expert ATS parser.
    Extract candidate details from the following CV text and return in structured JSON format.
    If information is missing, put "Not provided" or empty list [].

    CV Text:
    {cv_text}

    {format_instructions}
    """)

    chain = prompt | llm | parser

    fallback = {
        "name": "Not provided",
        "email": "Not provided",
        "phone": "Not provided",
        "location": "Not provided",
        "years_experience": 0,
        "skills": [],
        "education": [],
        "key_projects": [],
        "validation_status": "incomplete",
        "cv_filename": cv_filename
    }

    try:
        result = chain.invoke({"cv_text": clean_text, "format_instructions": format_instructions})
        result["cv_filename"] = cv_filename
        return result
    except Exception as e:
        print(f"⚠️ Parsing failed: {e}")
        return fallback

In [ ]:
cv_filename = os.path.basename(cv_files[1])
profile = generate_candidate_profile(cv_text, cv_filename)
display(profile)

/tmp/ipython-input-895771275.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


{'name': 'Mahmoud Nasser',
 'email': 'mahmoud.nasser.abdelfatah@gmail.com',
 'phone': '01119065308',
 'location': 'Giza, Egypt',
 'years_experience': '2',
 'skills': 'Python, SQL, C++, Java, C#, Streamlit, Flask, TensorFlow, PyTorch, MLflow, Decision Trees, Random Forests, Neural Networks, DNNs, CNNs, RNNs, Data Cleaning, Preprocessing, Feature Engineering, SMOTE, Time Series Analysis, Plotly, Matplotlib, API Integration, MLOps, Prompt Engineering, LLMs, GANs, Computer Vision, Speech-to-Text, Real-time AI integrations, Problem-Solving, Attention to Detail, Time Management, Team Collaboration, Communication',
 'education': 'B.Sc. in Artificial Intelligence, Faculty of Computers and Artificial Intelligence, Helwan University (09/2021 – 06/2025), Grade: Very Good, Graduation Project: Quranic ASR System (Whisper + NLP, Arabic Phonetics) – Grade A+',
 'key_projects': 'Follow-Up My Quran – Full‑stack AI‑based Quran learning platform with Whisper ASR for recitation feedback; E‑Commerce Data A

In [ ]:
def generate_general_summary(profile: Dict[str, Any]) -> str:
    """Generate a professional, job-agnostic candidate summary based only on profile."""
    safe_profile = json.dumps(profile, ensure_ascii=False, indent=2)

    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-oss-120b",
        temperature=0,
        openai_api_key=openai_api_key
    )

    prompt = ChatPromptTemplate.from_template("""
    You are a professional recruiter writing a neutral, detailed candidate summary.
    Base your summary only on the given candidate profile JSON below.

    Candidate Profile:
    {profile}

    Requirements:
    - 120–180 words max
    - Professional tone (as if for a candidate search result)
    - Cover: name, years of experience, main skills, education, key projects
    - Avoid speculation or subjective phrases ("passionate", "enthusiastic")
    - No reference to job descriptions or matching to a role
    """)

    chain = prompt | llm

    try:
        result = chain.invoke({"profile": safe_profile})
        summary = result.content.strip()

        if not summary:
            summary = (
                f"{profile.get('name','Unknown')} has {profile.get('years_experience',0)} years "
                f"experience and skills: {', '.join(profile.get('skills', []))}."
            )

        return summary
    except Exception as e:
        print(f"⚠️ Summary generation failed: {e}")
        return f"{profile.get('name','Unknown')} - summary not generated."

In [ ]:
contextual_summary = generate_general_summary(profile)

print("=== Candidate Contextual Summary ===")
print(contextual_summary)

=== Candidate Contextual Summary ===
Mahmoud Nasser is a junior AI/ML engineer with two years of professional experience. He holds a B.Sc. in Artificial Intelligence from the Faculty of Computers and Artificial Intelligence, Helwan University (2021‑2025), graduating with a “Very Good” grade and an A+ on his Quranic Automatic Speech Recognition (ASR) project (Whisper + NLP, Arabic phonetics). His technical skill set includes Python, SQL, C++, Java, C#, Streamlit, Flask, TensorFlow, PyTorch, MLflow, XGBoost, decision‑tree ensembles, deep‑learning architectures (CNN, RNN, DNN), data cleaning, feature engineering, SMOTE, time‑series analysis, visualization (Plotly, Matplotlib), API integration, MLOps, prompt engineering, LLMs, GANs, computer‑vision, and speech‑to‑text pipelines. Notable projects comprise: “Follow‑Up My Quran,” a full‑stack AI‑driven Quran learning platform with Whisper ASR feedback; an Azure‑SQL‑to‑Power BI e‑commerce analytics and predictive pipeline using Vanna AI and XG

In [ ]:
import re
from typing import Optional

def extract_professional_summary(cv_text: str) -> Optional[str]:
    """
    Extract professional summary or profile section from CV text.
    Looks for headings like 'Professional Summary', 'Summary', or 'Profile'
    and captures until the next section heading (Skills, Experience, Education, Projects) or end of text.
    """
    pattern = r"(?:Professional Summary|Summary|Profile)\s*(.*?)(?=\n(?:Skills|Experience|Education|Projects|\w+\s*:\s*)|\Z)"
    match = re.search(pattern, cv_text, flags=re.I | re.S)
    if match:
        return match.group(1).strip()
    return None

In [ ]:
extract_professional_summary(cv_text)

'AI Developer & Machine Learning Engineer with hands-on experience in building intelligent systems using \nLLMs, GANs, and deep learning techniques. Skilled in Python, NLP, computer vision, and real-time AI \nintegrations. Developed and deployed full-stack AI applications including chatbot agents, speech-to-text \nsystems, and text-to-image models. Strong background in prompt engineering, MLOps, and statistical \nmodeling. Passionate about solving real-world problems through scalable, intelligent solutions.'

In [ ]:
def process_cvs(cv_files: List[str]) -> List[Dict[str, Any]]:
    """
    Process CVs -> extract profile -> generate LLM summary -> merge summaries -> cache results.
    Stores combined summaries and full metadata for vector search and later usage.
    """

    processed_candidates = []

    for cv_file in cv_files:
        # Use CV text hash for caching (not job description)
        cv_text = extract_text_from_cv(cv_file)
        content_hash = hashlib.md5(cv_text.encode()).hexdigest()
        cache_file = f"/content/cache_{content_hash}.json"

        if os.path.exists(cache_file):
            with open(cache_file, "r", encoding="utf-8") as f:
                candidate_data = json.load(f)
            print(f"✅ Loaded cached data for {cv_file}")

        else:
            print(f"🔄 Processing {cv_file}...")
            try:
                # 1. Generate candidate profile (name, email, phone, etc.)
                profile = generate_candidate_profile(cv_text, cv_file)

                # 2. Extract professional summary (if explicitly written in CV)
                extracted_summary = extract_professional_summary(cv_text)

                # 3. Generate general LLM summary (without job description)
                try:
                    llm_summary = generate_general_summary(profile)
                except Exception as e:
                    print(f"⚠️ LLM summary generation failed: {e}")
                    llm_summary = extracted_summary or "No summary available."

                # 4. Merge extracted + LLM summary
                combined_summary = ""
                if extracted_summary:
                    combined_summary += f"PROFILE SUMMARY (from CV):\n{extracted_summary}\n\n"
                combined_summary += f"AI-GENERATED SUMMARY:\n{llm_summary}"

                # 5. Build final candidate data
                candidate_data = {
                    "page_content": combined_summary,
                    "metadata": {
                        "name": profile.get("name", "Unknown"),
                        "email": profile.get("email", "Not provided"),
                        "phone": profile.get("phone", "Not provided"),
                        "location": profile.get("location", "Not provided"),
                        "skills": profile.get("skills", []),
                        "experience_years": profile.get("experience_years", "Unknown"),
                        "cv_file_path": cv_file,
                        "validation_status": profile.get("validation_status", "unknown"),
                    },
                    "raw_text": cv_text,
                    "profile": profile,  # keep full structured profile for later use
                }

                # 6. Cache result
                with open(cache_file, "w", encoding="utf-8") as f:
                    json.dump(candidate_data, f, ensure_ascii=False, indent=2)
                print(f"💾 Cached result for {cv_file}")

            except Exception as e:
                print(f"❌ Error processing {cv_file}: {str(e)}")
                continue

        processed_candidates.append(candidate_data)

    return processed_candidates

In [ ]:
# === TEST CODE ===

# 3. Call the process_cvs function
print("=== Running process_cvs ===")
results = process_cvs(cv_files)

# 4. Print the results nicely
import pprint
pp = pprint.PrettyPrinter(indent=2)
for idx, candidate in enumerate(results, 1):
    print(f"\n--- Candidate {idx} ---")
    pp.pprint(candidate)

# 5. Optionally save results for later inspection
with open("processed_candidates.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nTest completed. Results saved to processed_candidates.json")

=== Running process_cvs ===
🔄 Processing /content/CVs/Jane Smith.pdf...
💾 Cached result for /content/CVs/Jane Smith.pdf
🔄 Processing /content/CVs/Mahmoud Nasser CV.pdf...
💾 Cached result for /content/CVs/Mahmoud Nasser CV.pdf
🔄 Processing /content/CVs/Michael Brown.pdf...
💾 Cached result for /content/CVs/Michael Brown.pdf
🔄 Processing /content/CVs/John Doe.pdf...
💾 Cached result for /content/CVs/John Doe.pdf

--- Candidate 1 ---
{ 'metadata': { 'cv_file_path': '/content/CVs/Jane Smith.pdf',
                'email': 'jane.smith@email.com',
                'experience_years': 'Unknown',
                'location': 'AI City, AC',
                'name': 'Jane Smith',
                'phone': '(234) 567-8901',
                'skills': 'Python, PyTorch, TensorFlow, C++, SQL, Hugging '
                          'Face, GANs, VAEs, Transformers, Stable Diffusion, '
                          'AWS, Docker, Kubernetes, Git, MLflow, Project '
                          'leadership, cross-functiona

### Vector Database & Retrieval

In [ ]:
# ---- Embeddings ----
embedding_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)
llm_embeddings = SentenceTransformerEmbeddings(model_name=embedding_model_name)

/tmp/ipython-input-2286231087.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  llm_embeddings = SentenceTransformerEmbeddings(model_name=embedding_model_name)


In [ ]:
# ---- Global Store for BM25 ----
bm25_corpus = []     # Tokenized summaries
bm25_docs = []       # Document objects
bm25 = None          # BM25 index


def build_vector_store(candidates: List[Dict[str, Any]]) -> FAISS:
    """
    Build FAISS (semantic) + BM25 (keyword) hybrid index from candidate data.
    Falls back gracefully if 'summary' missing by using page_content or profile info.
    """
    global bm25, bm25_corpus, bm25_docs
    documents = []
    bm25_corpus = []
    bm25_docs = []

    for candidate in candidates:
        summary = candidate.get("summary", "")

        # ---- Fallback: use page_content AI-GENERATED SUMMARY if no summary ----
        if not summary and "page_content" in candidate:
            text = candidate["page_content"]
            if "AI-GENERATED SUMMARY:" in text:
                # Extract only the AI-GENERATED SUMMARY section
                summary = text.split("AI-GENERATED SUMMARY:")[-1].strip()
            else:
                summary = text.strip()

        # ---- Fallback: use profile info if still empty ----
        if not summary:
            profile = candidate.get("profile", {})
            summary = f"{profile.get('skills', '')}. {profile.get('key_projects', '')}".strip()

        if not summary:
            continue  # Skip if no usable text at all

        metadata = candidate.get("metadata", {})
        if "name" not in metadata:
            metadata["name"] = candidate.get("profile", {}).get("name", "Unknown")

        doc = Document(page_content=summary, metadata=metadata)
        documents.append(doc)

        # Prepare BM25 corpus
        bm25_docs.append(doc)
        bm25_corpus.append(summary.lower().split())

    if not documents:
        raise ValueError("❌ No valid candidate summaries found after applying all fallbacks.")

    # Build FAISS store
    vector_store = FAISS.from_documents(documents, embedding=llm_embeddings)

    # Build BM25 index
    bm25 = BM25Okapi(bm25_corpus)

    print(f"✅ Built vector store with {len(documents)} candidates.")
    return vector_store


def query_candidates(vector_store: FAISS, job_description: str, top_n: int = 5) -> List[Dict[str, Any]]:
    """
    Perform Hybrid Search:
    1. Semantic Search with FAISS
    2. Keyword Search with BM25
    3. Weighted combination of scores
    """
    global bm25, bm25_docs

    if bm25 is None or not bm25_docs:
        raise ValueError("BM25 index not built. Call build_vector_store first.")

    # --- Semantic Search ---
    query_vector = embedding_model.encode(job_description)
    scores, indices = vector_store.index.search(np.array([query_vector]).astype("float32"), len(bm25_docs))

    semantic_results = []
    for rank, idx in enumerate(indices[0]):
        doc = vector_store.docstore.search(vector_store.index_to_docstore_id[idx])
        semantic_results.append({
            "doc": doc,
            "semantic_score": 1 - scores[0][rank]  # Convert distance → similarity
        })

    # --- BM25 Search ---
    tokenized_query = job_description.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1e-6

    # --- Combine Scores ---
    combined = []
    for i, doc in enumerate(bm25_docs):
        semantic_score = next((r["semantic_score"] for r in semantic_results if r["doc"] == doc), 0.0)
        keyword_score = bm25_scores[i]

        final_score = 0.65 * semantic_score + 0.35 * (keyword_score / max_bm25)
        combined.append({
            "doc": doc,
            "final_score": final_score,
            "semantic_score": semantic_score,
            "keyword_score": keyword_score
        })

    combined.sort(key=lambda x: x["final_score"], reverse=True)
    combined = combined[:top_n]

    results = []
    for rank, item in enumerate(combined):
        doc = item["doc"]
        results.append({
            "rank": rank + 1,
            "name": doc.metadata.get("name", "Unknown"),
            "email": doc.metadata.get("email", "Not provided"),
            "phone": doc.metadata.get("phone", "Not provided"),
            "similarity_score": float(item["final_score"]),
            "semantic_score": float(item["semantic_score"]),
            "keyword_score": float(item["keyword_score"]),
            "summary": doc.page_content,
            "validation_status": doc.metadata.get("validation_status", "incomplete")
        })

    return results

In [ ]:
def rerank_candidates(candidates: List[Dict], job_description: str) -> List[Dict]:
    """
    Re-rank candidates using LLM for nuanced contextual matching.
    Uses hybrid search results as input and asks LLM to produce the optimal order.
    Falls back gracefully if LLM fails or returns invalid output.
    """
    if not candidates or len(candidates) <= 1:
        return candidates  # No need to rerank if 0 or 1 candidate

    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        model_name="openai/gpt-oss-120b",
        temperature=0,
        openai_api_key=openai_api_key
    )

    # Create compact list of candidates with name + summary preview
    candidate_list_str = "\n".join([
        f"{i+1}. Name: {c.get('name', 'Unknown')} | Summary: {c.get('summary', '')[:300]}..."
        for i, c in enumerate(candidates)
    ])

    prompt_template = PromptTemplate(
        input_variables=["candidates", "job_description"],
        template="""
        You are an HR expert. Re-rank these candidates from best fit to worst fit
        for the given job description based on skills, experience, and relevance.

        Job Description:
        {job_description}

        Candidates:
        {candidates}

        Return ONLY a JSON array of candidate indices (1-based) in best-to-worst order.
        Example: [3, 1, 2, 5, 4]
        """
    )

    chain = LLMChain(llm=llm, prompt=prompt_template)

    try:
        result = chain.run(candidates=candidate_list_str, job_description=job_description)

        # Ensure clean JSON extraction
        result = result.strip()
        if not result.startswith("["):
            result = result[result.find("["):]  # Try to recover JSON part only
        ranked_indices = json.loads(result)

        if isinstance(ranked_indices, list) and all(
            isinstance(i, int) and 1 <= i <= len(candidates) for i in ranked_indices
        ):
            # Reorder candidates based on LLM output
            return [candidates[i - 1] for i in ranked_indices]

        print("⚠️ Invalid indices returned by LLM. Returning original order.")
        return candidates

    except Exception as e:
        print(f"⚠️ Reranking failed: {e}. Returning original order.")
        return candidates

In [ ]:
print("\n🚀 Starting Candidate Search Test")

# 1. Build Vector Store
print("\n=== Building Vector Store ===")
vector_store = build_vector_store(results)

# 2. Query Top Candidates
print("\n=== Querying Top Candidates ===")
top_candidates = query_candidates(vector_store, job_description, top_n=2)

# 3. Show Results BEFORE Re-Rank
print("\n=== Top Candidates (Before Re-Rank) ===")
for candidate in top_candidates:
    print(f"• Rank {candidate['rank']}: {candidate['name']} | Similarity: {candidate['similarity_score']:.4f}")

# 4. Re-Rank Candidates with LLM
print("\n=== Applying LLM Re-Rank ===")
reranked_candidates = rerank_candidates(top_candidates, job_description)

# 5. Show Results AFTER Re-Rank
print("\n=== Top Candidates (After Re-Rank) ===")
for i, candidate in enumerate(reranked_candidates, start=1):
    print(f"• Rank {i}: {candidate['name']} | Similarity: {candidate['similarity_score']:.4f}")

# 6. Save Final Results
output_file = "processed_candidates.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(reranked_candidates, f, indent=2, ensure_ascii=False)

print(f"\n✅ Candidate search & re-rank complete. Results saved to {output_file}")


🚀 Starting Candidate Search Test

=== Building Vector Store ===
✅ Built vector store with 4 candidates.

=== Querying Top Candidates ===

=== Top Candidates (Before Re-Rank) ===
• Rank 1: John Doe | Similarity: 0.3500
• Rank 2: Dr. Michael Brown | Similarity: 0.3064

=== Applying LLM Re-Rank ===


/tmp/ipython-input-498485885.py:40: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)
/tmp/ipython-input-498485885.py:43: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = chain.run(candidates=candidate_list_str, job_description=job_description)



=== Top Candidates (After Re-Rank) ===
• Rank 1: John Doe | Similarity: 0.3500
• Rank 2: Dr. Michael Brown | Similarity: 0.3064

✅ Candidate search & re-rank complete. Results saved to processed_candidates.json


In [ ]:
reranked_candidates

[{'rank': 1,
  'name': 'John Doe',
  'email': 'john.doe@email.com',
  'phone': '(123) 456-7890',
  'similarity_score': 0.35,
  'semantic_score': 0.0,
  'keyword_score': 4.3673250574895786,
  'summary': 'John Doe is a recent graduate (B.Sc. Computer Science, Tech University, May\u202f2025) with zero years of professional experience. His technical skill set includes Python, C++, Java, PyTorch, TensorFlow, Hugging\u202fFace, OpenAI API, generative models (GANs, VAEs, Stable Diffusion), Docker, Git, Jupyter, and basic AWS, complemented by teamwork, problem‑solving, and technical communication abilities. In his final academic year, John led a Text‑to‑Image Generator project (Jan–May\u202f2025) that implemented a Stable Diffusion model, achieving a 15\u202f% improvement in FID score and presenting the results at the university AI symposium. He also developed a transformer‑based chatbot with integrated sentiment analysis (Sep–Dec\u202f2024), reaching 85\u202f% accuracy and deploying the syste

### Email Draft Generation

In [ ]:
def generate_recruitment_emails(candidate: Dict[str, Any], job_description: str,
                                company_name: str = "Your Company",
                                sender_role: str = "HR Manager") -> str:
    """Generate personalized recruitment email with company signature."""

    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-oss-120b",
        temperature=0.6,  # قللناها عشان يطلع نص ثابت أكثر
        openai_api_key=openai_api_key
    )

    prompt_template = PromptTemplate(
    input_variables=["name", "skills", "experience", "summary", "job_description", "sender_role", "company_name"],
    template="""
Write a short, professional, and friendly recruitment email in **HTML format** to the candidate below.

Candidate Information:
- Name: {name}
- Key Skills: {skills}
- Experience: {experience} years
- Profile Summary: {summary}

Job Description:
{job_description}

Instructions:
- Use <p> tags for paragraphs and <br> for line breaks where needed.
- Greet the candidate by name in the first paragraph.
- Thank them for applying.
- Mention at least one specific match between their skills/experience and the job.
- Invite them to the next step (interview or call).
- Keep it warm, polite, and professional.
- Limit to 2-3 short paragraphs.
- Do NOT include personal email addresses.
- End with a professional signature in this format:
  <p>Best regards,<br><b>{sender_role}</b><br>{company_name}</p>
"""
)

    # Extract profile info safely
    profile = candidate.get('profile', {})
    name = candidate.get('name') or profile.get('name', 'Candidate')
    skills_list = profile.get('skills', [])
    skills = ', '.join(skills_list) if skills_list else 'Not specified'
    experience = profile.get('years_experience', 0)
    summary = candidate.get('summary', 'No summary provided')

    chain = LLMChain(llm=llm, prompt=prompt_template)
    email_content = chain.run(
        name=name,
        skills=skills,
        experience=experience,
        summary=summary,
        job_description=job_description,
        company_name=company_name,
        sender_role=sender_role
    )

    return email_content

In [ ]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import smtplib

def send_emails(email_content: str, recipient: str, sender_email: str,
                subject: str = "Job Opportunity", app_password: str = None,
                use_html: bool = True) -> bool:
    """
    Send an email via Gmail SMTP using App Password stored in Colab userdata (HR_SMTP_GMAIL).
    Supports HTML or plain text emails.
    Returns True if the email was sent successfully, False otherwise.
    """
    try:
        # Get Gmail App Password (priority: parameter > userdata)
        app_password = app_password or userdata.get("HR_SMTP_GMAIL")
        if not app_password:
            print("❌ No Gmail app password found. Please set HR_SMTP_GMAIL or pass it explicitly.")
            return False

        # Build email message
        msg = MIMEMultipart()
        msg["From"] = sender_email
        msg["To"] = recipient
        msg["Subject"] = subject

        mime_type = "html" if use_html else "plain"
        msg.attach(MIMEText(email_content, mime_type))

        print(f"📧 Connecting to Gmail SMTP as: {sender_email}")
        print(f"➡ Sending email to: {recipient} | Subject: {subject}")

        # Connect to Gmail SMTP
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(sender_email, app_password)
            server.send_message(msg)

        print(f"✅ Email successfully sent to {recipient}")
        return True

    except smtplib.SMTPAuthenticationError as auth_err:
        print("❌ Authentication error: Check email & app password. Ensure 2FA is enabled.")
        print(f"Details: {auth_err}")
        return False

    except smtplib.SMTPConnectError as conn_err:
        print("❌ Connection error: Could not connect to Gmail SMTP.")
        print(f"Details: {conn_err}")
        return False

    except Exception as e:
        print(f"❌ Failed to send email to {recipient}: {e}")
        return False

In [ ]:
# === EXAMPLE USAGE WITH TEST CODE ===
for candidate in reranked_candidates:  # reranked from your test code
    email_content = generate_recruitment_emails(candidate, job_description, company_name="New Age Tech.")
    recipient_email = candidate.get('email') or candidate.get('profile', {}).get('email')
    if recipient_email and recipient_email != "Not provided":
        send_emails(email_content, recipient_email, sender_email="mhnasser00@gmail.com")
    else:
        print(f"⚠️ Skipping candidate {candidate.get('name', 'Unknown')} due to missing email.")

📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: john.doe@email.com | Subject: Job Opportunity
✅ Email successfully sent to john.doe@email.com
📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: michael.brown@email.com | Subject: Job Opportunity
✅ Email successfully sent to michael.brown@email.com


In [ ]:
# ================== FULL PIPELINE ==================
def full_pipeline(results: List[Dict[str, Any]], job_description: str,
                  top_n: int = 1, sender_email: str = "mhnasser00@gmail.com",
                  company_name: str="New Age Tech.", sender_role: str = "HR Manager"):
    print("\n🚀 Starting Candidate Search & Email Pipeline")

    print("\n=== Building Vector Store ===")
    vector_store = build_vector_store(results)

    print("\n=== Hybrid Search (BM25 + FAISS) ===")
    top_candidates = query_candidates(vector_store, job_description, top_n=top_n)

    print("\n=== BEFORE RERANK ===")
    for c in top_candidates:
        print(f"{c['rank']}. {c['name']} | Similarity: {c['similarity_score']:.4f}")

    print("\n=== RERANKING WITH LLM ===")
    reranked = rerank_candidates(top_candidates, job_description)

    print("\n=== AFTER RERANK ===")
    for i, c in enumerate(reranked):
        print(f"{i+1}. {c['name']} | Similarity: {c['similarity_score']:.4f}")

    print("\n=== GENERATING & SENDING EMAILS ===")
    for candidate in reranked:
        email_content = generate_recruitment_emails(candidate, job_description, company_name, sender_role)
        recipient = candidate.get("email")
        if recipient and recipient != "Not provided":
            send_emails(email_content, recipient, sender_email = sender_email)
        else:
            print(f"⚠️ No email found for {candidate['name']} – Skipping.")

    with open("processed_candidates.json", "w", encoding="utf-8") as f:
        json.dump(reranked, f, indent=2, ensure_ascii=False)

    print("\n✅ Pipeline completed. Results saved to processed_candidates.json")

In [ ]:
job_description = """
We are hiring a Junior Generative AI Engineer (0-1 year experience).
Responsibilities: assist in developing, fine-tuning, and deploying generative AI models (LLMs, diffusion models),
conduct experiments with open-source models, prepare and clean datasets, collaborate with engineers to integrate models,
and stay updated on AI research.
Requirements: Bachelor's degree in Computer Science, AI, or related field, understanding of machine learning fundamentals,
knowledge of deep learning frameworks (PyTorch or TensorFlow), Python programming skills, data manipulation (NumPy, Pandas),
and willingness to learn and experiment with LLMs and prompt engineering.
Nice to have: exposure to Hugging Face, LangChain, cloud platforms (AWS, GCP, Azure), and MLOps basics.
"""

full_pipeline(results, job_description, top_n=3)


🚀 Starting Candidate Search & Email Pipeline

=== Building Vector Store ===
✅ Built vector store with 4 candidates.

=== Hybrid Search (BM25 + FAISS) ===

=== BEFORE RERANK ===
1. Mahmoud Nasser | Similarity: 0.3500
2. John Doe | Similarity: 0.2677
3. Jane Smith | Similarity: 0.2333

=== RERANKING WITH LLM ===

=== AFTER RERANK ===
1. John Doe | Similarity: 0.2677
2. Mahmoud Nasser | Similarity: 0.3500
3. Jane Smith | Similarity: 0.2333

=== GENERATING & SENDING EMAILS ===
📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: john.doe@email.com | Subject: Job Opportunity
✅ Email successfully sent to john.doe@email.com
📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: mahmoud.nasser.abdelfatah@gmail.com | Subject: Job Opportunity
✅ Email successfully sent to mahmoud.nasser.abdelfatah@gmail.com
📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: jane.smith@email.com | Subject: Job Opportunity
✅ Email successfully sent to jane.s

In [ ]:
job_description = """
We are hiring a Junior Data Scientist (0-1 year experience).
Responsibilities: collect, clean, and analyze datasets to generate actionable insights,
build and evaluate machine learning models for predictive and descriptive analytics,
create data visualizations and reports, collaborate with engineers and business teams to deliver data-driven solutions,
and stay updated on the latest trends in data science and machine learning.
Requirements: Bachelor's degree in Computer Science, Statistics, Data Science, or related field,
understanding of statistics and machine learning fundamentals, proficiency in Python (NumPy, Pandas, Scikit-learn),
experience with data visualization tools (Matplotlib, Seaborn, or Plotly), and strong problem-solving skills.
Nice to have: exposure to SQL, big data tools (Spark, Hadoop), cloud platforms (AWS, GCP, Azure),
and experience with version control (Git) and basic MLOps practices.
"""

results = process_cvs(cv_files)

full_pipeline(results, job_description, top_n=3)

✅ Loaded cached data for /content/CVs/Jane Smith.pdf
✅ Loaded cached data for /content/CVs/Mahmoud Nasser CV.pdf
✅ Loaded cached data for /content/CVs/Michael Brown.pdf
🔄 Processing /content/CVs/Mustafa_Nasser_Data Science_00.pdf.pdf.pdf...
💾 Cached result for /content/CVs/Mustafa_Nasser_Data Science_00.pdf.pdf.pdf
✅ Loaded cached data for /content/CVs/John Doe.pdf

🚀 Starting Candidate Search & Email Pipeline

=== Building Vector Store ===
✅ Built vector store with 5 candidates.

=== Hybrid Search (BM25 + FAISS) ===

=== BEFORE RERANK ===
1. Mustafa Nasser Abdel-Fatah | Similarity: 0.3500
2. Mahmoud Nasser | Similarity: 0.3440
3. Jane Smith | Similarity: 0.3208

=== RERANKING WITH LLM ===

=== AFTER RERANK ===
1. Mahmoud Nasser | Similarity: 0.3440
2. Jane Smith | Similarity: 0.3208
3. Mustafa Nasser Abdel-Fatah | Similarity: 0.3500

=== GENERATING & SENDING EMAILS ===
📧 Connecting to Gmail SMTP as: mhnasser00@gmail.com
➡ Sending email to: mahmoud.nasser.abdelfatah@gmail.com | Subject

In [ ]:
vector_store = FAISS.load_local(
    "/content/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

retrieve_and_contact_candidates(vector_store=vector_store, job_description=job_description, send_email=True, sender_email="mhnasser00@gmail.com")


=== Top Candidates ===
1. John Doe | Email: john.doe@email.com | Similarity: 0.3152
Connecting to Gmail SMTP server as mhnasser00@gmail.com...
Logging in...
Email successfully sent to john.doe@email.com
📧 Email sent to john.doe@email.com


[{'rank': 1,
  'name': 'John Doe',
  'email': 'john.doe@email.com',
  'phone': '(123) 456-7890',
  'similarity_score': 0.31516826152801514,
  'validation_status': 'complete',
  'summary': 'John Doe is a recent Computer Science graduate with solid fundamentals in Python, PyTorch, TensorFlow, and C++, directly aligning with core ML‑engineer skill sets. His hands‑on experience includes building a Stable Diffusion‑based text‑to‑image generator (improving FID by 15%) and a GPT‑2 transformer chatbot with sentiment analysis (85% accuracy), demonstrating proficiency in generative models, Hugging Face, and API integration. He is comfortable with version control (Git), containerization (Docker), and basic AWS services, supporting scalable deployment and collaborative development. Strong problem‑solving, technical communication, and team collaboration abilities are evident from university symposium presentations and Flask web‑app delivery. The primary gap is limited professional work experience (